# Week 12 - 3D Vision: Stereo, Depth and Point Clouds

**MCTE 4323 / MCTA 4364 Machine Vision**

### Learning objectives
By the end of this lab you will be able to:
- Explain **projective geometry** and the role of the camera matrix in 3D.
- Compute a **disparity map** from a stereo pair and convert it to **depth**.
- Back-project depth to a **3D point cloud** using the intrinsics.
- Visualise and interpret point clouds for engineering tasks.

### Depth from stereo
Two cameras separated by a **baseline** $B$ see a point at different horizontal positions. The pixel difference (disparity) $d$ relates to depth $Z$ by

$$ Z = \frac{f \cdot B}{d} $$

Close object = large disparity; far object = small disparity.

## 1. Setup

In [ ]:
import os
if not os.path.isdir("MCTA-4364-Machine-Vision"):
    !git clone https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git
%cd MCTA-4364-Machine-Vision
!pip -q install opencv-python matplotlib numpy scipy ipywidgets

In [ ]:
import sys
sys.path.append("resources/scripts")
import cv2, numpy as np
from cvhelpers import show, concept_map
print("OpenCV:", cv2.__version__)

## 2. The 3D vision pipeline

In [ ]:
concept_map([
    "Calibrate cameras -> intrinsics K and baseline B",
    "Rectify the stereo pair (epipolar lines become horizontal)",
    "Compute disparity d(x, y) with StereoBM / SGBM",
    "Depth Z = f * B / d",
    "Back-project each pixel to a 3D point (X, Y, Z)",
    "Point cloud: visualise, segment, measure, register"
], title="From a stereo pair to a 3D point cloud")

## 3. Guided example - build a synthetic stereo pair
We render a random-dot scene with **two depth layers**. Matching these dots is the same problem a real stereo camera solves.

In [ ]:
h, w = 240, 320
rng = np.random.RandomState(0)
dots = (rng.rand(h, w) < 0.12).astype(np.uint8) * 255

# Two depth layers: left half is far (small disparity), right half is near (large disparity)
disp_gt = np.full((h, w), 8, np.float32)
disp_gt[:, w // 2:] = 24

left = np.zeros((h, w), np.uint8)
right = np.zeros((h, w), np.uint8)
ys, xs = np.where(dots > 0)
for y, x in zip(ys, xs):
    left[y, x] = 255
    xr = x - int(disp_gt[y, x])
    if 0 <= xr < w:
        right[y, xr] = 255

show(left, right, titles=["Left image", "Right image (shifted)"])

## 4. Guided example - disparity with Semi-Global Block Matching
`StereoSGBM` searches for the best matching block along the epipolar line. The result is a disparity map (in pixels, scaled by 16).

In [ ]:
num_disp = 32
block = 5
sgbm = cv2.StereoSGBM_create(minDisparity=0, numDisparities=num_disp, blockSize=block,
                             P1=8 * block * block, P2=32 * block * block,
                             uniquenessRatio=5, speckleWindowSize=50, speckleRange=2)
disp = sgbm.compute(left, right).astype(np.float32) / 16.0
disp[disp < 0] = 0

disp_vis = cv2.normalize(disp, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
show(left, disp_vis, titles=["Left image", "Disparity (brighter = nearer)"])

## 5. Guided example - disparity to depth, then to a point cloud
Using $f = 400$ px and baseline $B = 0.1$ m, we convert disparity to depth and back-project each pixel:
$$ X = \frac{(u-c_x)Z}{f},\qquad Y = \frac{(v-c_y)Z}{f},\qquad Z = \frac{fB}{d}.$$

In [ ]:
f = 400.0            # focal length in pixels
B = 0.1              # baseline in metres
cx, cy = w / 2, h / 2

# Convert disparity to depth (avoid divide by zero)
depth = np.zeros_like(disp)
valid = disp > 0
depth[valid] = f * B / disp[valid]
print("Depth range (valid):", round(float(depth[valid].min()), 3), "to",
      round(float(depth[valid].max()), 3), "m")

vv, uu = np.mgrid[0:h, 0:w]
X = (uu - cx) * depth / f
Y = (vv - cy) * depth / f
Z = depth

# Downsample for plotting
step = 2
pts = np.stack([X[::step, ::step], Y[::step, ::step], Z[::step, ::step]], axis=-1).reshape(-1, 3)
col = disp[::step, ::step].reshape(-1)
m = col > 0
pts, col = pts[m], col[m]

In [ ]:
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
p = ax.scatter(pts[:, 0], pts[:, 2], -pts[:, 1], c=col, cmap="viridis", s=4)
ax.set_xlabel("X (m)"); ax.set_ylabel("Z depth (m)"); ax.set_zlabel("-Y")
ax.set_title("Reconstructed point cloud (coloured by disparity)")
fig.colorbar(p, ax=ax, shrink=0.6, label="disparity (px)")
plt.tight_layout(); plt.show()

###  Interactive exploration - view the point cloud from different angles
Change elevation and azimuth to inspect the two depth layers. This is how engineers inspect a scan for defects or measurements.

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

def cloud_demo(elev=20, azim=-60, max_depth=4.0):
    fig = plt.figure(figsize=(7, 5))
    ax = fig.add_subplot(111, projection="3d")
    sel = pts[:, 2] < max_depth
    ax.scatter(pts[sel, 0], pts[sel, 2], -pts[sel, 1], c=col[sel], cmap="viridis", s=4)
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlabel("X"); ax.set_ylabel("Z"); ax.set_zlabel("-Y")
    plt.tight_layout(); plt.show()

interact(cloud_demo,
         elev=widgets.IntSlider(min=0, max=90, step=5, value=20),
         azim=widgets.IntSlider(min=-180, max=180, step=10, value=-60),
         max_depth=widgets.FloatSlider(min=0.5, max=6.0, step=0.5, value=4.0))

## 6. Optional - real point cloud with Open3D
Open3D is the standard library for point-cloud processing. `draw_plotly` renders interactively inside Colab.

In [ ]:
try:
    import open3d as o3d
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts.astype(np.float64))
    pcd.paint_uniform_color([0.1, 0.5, 0.9])
    o3d.visualization.draw_plotly([pcd])
except Exception as e:
    print("Open3D not available here (install in a local environment). Info:", e)

## 7. Exercise (complete the code)

1. Change the right-hand disparity to **40** and recompute the depth.
2. What happens to the reconstructed depth of that half? Verify with $Z = fB/d$.
3. Explain why depth becomes **noisier** as disparity gets smaller (objects far away).

In [ ]:
# TODO: regenerate with different disparity and compare depth


## 8. Challenge (independent)

Capture (or download) a real stereo pair from a stereo camera or a phone in two positions. Rectify, compute depth, and build a point cloud. Identify one source of error (wrong baseline, poor calibration, textureless regions) and explain its effect on depth.

In [ ]:
# Your code here


## 9. Check your understanding (Q&A)

<details><summary><b>Q1. Why does a larger baseline give more accurate depth?</b></summary>

A larger baseline produces a larger disparity for the same depth, so a small error in disparity causes a smaller error in depth. Depth error grows with $Z^2/(fB)$.
</details>

<details><summary><b>Q2. Why are textureless surfaces a problem for stereo?</b></summary>

Block matching relies on distinctive texture to find correspondences. A flat, uniform region can match anywhere, so disparity is undefined or wrong.
</details>

<details><summary><b>Q3. What is the difference between a depth map and a point cloud?</b></summary>

A depth map stores one depth value per pixel in image coordinates. A point cloud stores explicit 3D coordinates (X, Y, Z) per point and can be viewed and processed in 3D space.
</details>

## 10. Further reading & self-exploration
- OpenCV depth maps from stereo: https://docs.opencv.org/4.x/dd/d53/tutorial_py_depthmap.html
- OpenCV stereo calibration and rectification: https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html
- Open3D tutorials: https://www.open3d.org/docs/release/tutorial/geometry/pointcloud.html
- Szeliski, *Computer Vision* (free) - 3D reconstruction: https://szeliski.org/Book/
- Wikipedia - Stereo vision: https://en.wikipedia.org/wiki/Computer_stereo_vision
- Wikipedia - Point cloud: https://en.wikipedia.org/wiki/Point_cloud

**Try next:** estimate depth from a single image using a monocular depth model (MiDaS) and compare with stereo.

## 11. Key takeaways
- Depth from stereo: $Z = fB/d$.
- Larger baseline or focal length improves depth accuracy.
- SGBM gives dense disparity but struggles on textureless regions.
- Back-projecting depth with intrinsics yields a 3D point cloud.
- Point clouds power inspection, measurement, robot grasping and SLAM.